# 17 · Smoke Detection IoT (Kaggle) – a cautionary dataset

**Utility for the City Brain: low** · Domain: fire sensors · Incident types: `fire`

> 62,630 readings of a cheap air-quality sensor with a 'fire alarm' label. Useful for value ranges, but its labels leak through the recording session: never train or set thresholds on it.

| | |
|---|---|
| **Source** | Kaggle *Smoke Detection Dataset* (deepcontractor), public GitHub mirror used |
| **Licence** | Kaggle dataset licence, unclear for redistribution: not included here |
| **In this repo** | No data. Outputs below were produced on the full download |
| **Get it** | https://www.kaggle.com/datasets/deepcontractor/smoke-detection-dataset (`smoke_detection_iot.csv`, saved as `smoke.csv`) |

Legend used in the last section: **Detect** = detection rules (`brain/src/detection/`), **Graph** = graph DB (`brain/src/graph/`), **Mock** = realistic replay for mock sensors, **Scenarios** = test / demo cases, **Eval** = measure the Brain on data not used to tune it, **Score** = numbers for the scoring model.

In [1]:
import sys, json, gzip, re
from pathlib import Path
sys.path.insert(0, '..')
from cbdata import path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 160)
plt.rcParams.update({'figure.figsize': (10, 3.6), 'axes.grid': True, 'grid.alpha': .3})

## 1. What's inside

1 Hz readings: `UTC`, temperature, humidity, TVOC (ppb), eCO2 (ppm), raw H2, raw ethanol, pressure (hPa), PM1.0, PM2.5, particle counts NC0.5 / NC1.0 / NC2.5, `CNT` (a sample counter: **it leaks the label**, drop it), `Fire Alarm` (0/1).

In [2]:
s = pd.read_csv(path('air-quality/smoke-iot/smoke.csv'), index_col=0).sort_values('UTC')
print(f"{len(s):,} rows; label 'fire' = {s['Fire Alarm'].mean() * 100:.1f} % of rows"); s.head()

62,630 rows; label 'fire' = 71.5 % of rows


,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT,Fire Alarm
51142,1654712187,27.45,43.27,48,488,12844,20723,937.586,2.04,2.12,14.05,2.191,0.049,0,0
51143,1654712188,27.41,43.54,32,457,12857,20743,937.589,2.16,2.24,14.83,2.313,0.052,1,0
51144,1654712189,27.36,43.76,34,455,12857,20747,937.604,2.19,2.28,15.07,2.350,0.053,2,0
51145,1654712190,27.32,43.84,29,454,12858,20752,937.610,2.24,2.32,15.39,2.400,0.054,3,1
51146,1654712191,27.27,43.98,28,456,12860,20751,937.601,2.26,2.35,15.58,2.429,0.055,4,1


## 2. Why the labels can't be trusted

In [3]:
gap = s.UTC.diff() > 3600
print('Recording sessions (gaps > 1 h):', int(gap.sum()) + 1, '· label changes:', int(s['Fire Alarm'].diff().abs().sum()))
print('Temperature range in "fire" rows:', s.loc[s['Fire Alarm'] == 1, 'Temperature[C]'].agg(['min', 'max']).round(1).tolist())
best = max(((c, th, ((s[c] > th) == s['Fire Alarm']).mean()) for c in ['Pressure[hPa]', 'TVOC[ppb]', 'PM2.5', 'Humidity[%]']
            for th in s[c].quantile(np.linspace(.02, .98, 49)).unique()), key=lambda z: z[2])
print(f"Best single-feature rule: {best[0]} > {best[1]:.1f} -> accuracy {best[2] * 100:.0f} %")

Recording sessions (gaps > 1 h): 4 · label changes: 10
Temperature range in "fire" rows: [-22.0, 41.4]
Best single-feature rule: Pressure[hPa] > 937.6 -> accuracy 88 %


The best single feature is **pressure**, which cannot detect fire: the label mostly tells *which session* a row comes from, not whether there is a fire. Any model trained on it learns the session.

## 3. Useful value ranges

In [4]:
s.groupby('Fire Alarm')[['TVOC[ppb]', 'eCO2[ppm]', 'PM2.5', 'Humidity[%]']].quantile([.5, .9]).unstack().round(1).T

Fire Alarm            0       1
TVOC[ppb]   0.5    86.0  1103.0
            0.9  9007.8  1343.0
eCO2[ppm]   0.5   400.0   404.0
            0.9   454.0   643.0
PM2.5       0.5     1.5     1.9
            0.9     2.3     2.5
Humidity[%] 0.5    46.1    51.4
            0.9    54.2    55.3

## 4. What it means for the City Brain

| Use | How |
|---|---|
| **Mock** | Realistic noisy ranges for a cheap `air_quality` sensor (TVOC baseline ≈ 100 ppb with spikes to thousands **without** fire) |
| **Detect** | Confirms one air-quality reading alone must not open `fire`; use the NIST rule (notebook 10) |
| **Don't** | Train a model or set thresholds on it; say so in the report if we cite it |

**Limits:** 5 sessions, labels in long blocks, 71.5 % "fire" rows (the reverse of real life).